# ICU Mortality Prediction Model — **Phase 1b**
### All-admissions base (index + non-index), for the Phase 5 sensitivity analysis

*This notebook generalizes the patient identity resolution and sheet integration already validated and
tested in Phase 1 (Sections 4 and 6) and Phase 2 (Sections 2, 3, 4, and 5), removing only the step that
restricted the cohort to one admission per patient (the index admission). The result is the ~7,541
admissions in the database (minus exact duplicates and the few cases that remain ambiguous even after
positional recovery), instead of the 5,676 index admissions.*

**What changes relative to Phase 1/2, and what does not:**
- **Unchanged**: identity resolution via connected components (`patient_id`), propagation of duplicate
  removal to all sheets, positional-alignment recovery of ambiguous pairs, Folha8 integration by
  position, comorbidity grouping, and missingness indicators.
- **Changed**: Phase 1 collapsed the table to one row per patient (`drop_duplicates(subset='patient_id',
  keep='first')`) before integrating the other sheets. Here that step is removed — all of a patient's
  admissions remain, each carrying its own repeated `patient_id`.
- **Required order**: run after Phase 1 is complete and before Phase 2. The file produced
  (`modelo_A_todas_internacoes.csv`) is read by Phase 5, Section 6 (sensitivity analysis).

## 0 · Configuration and loading

> **Reproducibility note.** The original study ran in Google Colab with the source spreadsheet on Google Drive. In this repository, place the input file in `data/` and adjust `FILE_PATH` accordingly.

In [ ]:
import os
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60)

# === Configuration — identical to Phase 1 ===
FILE_PATH  = 'data/Excel01GlobalAdmission.xlsx'
FASE1B_DIR = 'outputs/phase1b'
OUTDIR     = FASE1B_DIR
os.makedirs(FASE1B_DIR, exist_ok=True)

MERGE_KEYS = ['EpimedCode', 'MedicalRecord']

AMBIGUOUS_POLICY = 'exclude'

SHEETS = {
    'admissao':     'Folha2',
    'comorbidades': 'Folha3',
    'saps3_razoes': 'Folha4',
    'complic_dia1': 'Folha5',
    'fisiologia6h': 'Folha6',
    'recursos_uti': 'Folha7',
    'diag_secund':  'Folha8',
}
print('Configuration loaded.')

In [ ]:
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f'File not found at:\n  {FILE_PATH}')

raw = {nome: pd.read_excel(FILE_PATH, sheet_name=folha, header=1)
       for nome, folha in SHEETS.items()}

for nome, df in raw.items():
    df['_linha_bruta'] = df.index

for nome, df in raw.items():
    print(f'{nome:14s} ({SHEETS[nome]}): {df.shape[0]:>5} linhas x {df.shape[1]:>3} colunas')


## 1 · Type cleaning, outcomes, and exact-duplicate removal

Identical to Phase 1 in every respect:
exact-duplicate removal is **propagated to all sheets** by original row position (`_linha_bruta`),
with key verification before each removal. This prevents copies from remaining in the predictor
sheets with different clinical content — which would cause incorrect association of clinical data
in the later merge.

In [ ]:
BOOL_MAP = {'verdadeiro':1,'falso':0,'sim':1,'no':0,'nao':0,'true':1,'false':0}

def to_bool(series):
    s = series.astype('string').str.strip().str.lower()
    return s.map(BOOL_MAP).astype('Float64')

def clean_booleans(df, protect):
    out = df.copy()
    for col in out.columns:
        if col in protect: continue
        vals = out[col].dropna().astype('string').str.strip().str.lower().unique()
        if len(vals) and set(vals).issubset(BOOL_MAP.keys()):
            out[col] = to_bool(out[col])
    return out

protect = set(MERGE_KEYS) | {'_linha_bruta'}
for nome in raw:
    raw[nome] = clean_booleans(raw[nome], protect)

adm = raw['admissao'].copy()

def derive_outcome(col):
    s = adm[col].astype('string').str.strip().str.lower()
    return s.map({'death':1,'obito':1,'alta':0}).astype('Float64')

adm['mortalidade_hospitalar'] = derive_outcome('HospitalDischargeName')
adm['mortalidade_uti']        = derive_outcome('UnitDischargeName')
adm['adm_datetime'] = pd.to_datetime(
    adm['UnitAdmissionDate'].astype(str) + ' ' + adm['Unit AdmissionTime'].astype(str),
    dayfirst=True, errors='coerce')

print(f'Raw admissions: {len(adm)}')


In [ ]:
n_antes = len(adm)
# _linha_bruta is unique by construction — it must stay out of the exact-duplicate check.
cols_comparar = [c for c in adm.columns if c != '_linha_bruta']

# Identify the exact copies to remove from Folha2 (always keep the 1st occurrence).
mask_dup = adm.duplicated(subset=cols_comparar, keep='first')
pos_remover = list(adm.loc[mask_dup, '_linha_bruta'])
chave_por_pos = {int(r['_linha_bruta']): (r['EpimedCode'], r['MedicalRecord'])
                 for _, r in adm.loc[mask_dup, ['_linha_bruta'] + MERGE_KEYS].iterrows()}
n_dup = int(mask_dup.sum())
adm = adm.loc[~mask_dup].reset_index(drop=True)
n_depois = len(adm)

# Propagate the SAME removal to the other sheets, with key verification per position.
# If the key in the sheet diverges from the Folha2 key at that position, removal
# is aborted for that position and the fact is reported — never silenced.
print('Exact-duplicate removal, propagated with per-position key verification:')
for nome in raw:
    if nome == 'admissao':
        print(f'  {nome:14s}: {n_dup} rows removed (Folha2, reference)')
        continue
    df = raw[nome]
    a_remover, inconsistentes = [], []
    for pos, chave in chave_por_pos.items():
        linha = df.loc[df['_linha_bruta'] == pos]
        if len(linha) == 1 and (linha['EpimedCode'].iloc[0], linha['MedicalRecord'].iloc[0]) == chave:
            a_remover.append(pos)
        else:
            inconsistentes.append(pos)
    raw[nome] = df.loc[~df['_linha_bruta'].isin(a_remover)].reset_index(drop=True)
    msg = f'  {nome:14s}: {len(a_remover)} linhas removidas'
    if inconsistentes:
        msg += (f' | ⚠️  {len(inconsistentes)} position(s) NOT removed due to divergent key '
                f'(content divergence between Folha2 and {nome}): positions {inconsistentes}')
    print(msg)

print(f'\nRaw admissions:              {n_antes}')
print(f'Duplicatas exatas removidas: {n_dup}')
print(f'Distinct admissions:         {n_depois}')


## 2 · Patient identity (`patient_id`) — identical to Phase 1, without collapsing to one row per patient

`patient_id` is built with the same connected components as Phase 1. The only difference is that here
**there is no collapse** to one row per patient — all admissions of the same patient keep the same
`patient_id`, which allows Phase 5's GroupKFold to guarantee that admissions of the same patient never
land in different folds.

In [ ]:
import networkx as nx

def patient_id_linked(df, keys=MERGE_KEYS):
    G = nx.Graph(); G.add_nodes_from(df.index)
    for col in keys:
        for _, idx in df.groupby(col).groups.items():
            idx = list(idx)
            for a in idx[1:]:
                G.add_edge(idx[0], a)
    comp = {n:i for i, c in enumerate(nx.connected_components(G)) for n in c}
    return df.index.map(comp).astype('int64')

adm['patient_id'] = patient_id_linked(adm)

n_pac = adm['patient_id'].nunique()
adm_por_pac = adm['patient_id'].value_counts()
print(f'Unique patients (connected components): {n_pac}')
print(f'  with 1 admission:      {(adm_por_pac==1).sum()}')
print(f'  with >1 admission:     {(adm_por_pac>1).sum()}')
print(f'  maximum admissions:    {adm_por_pac.max()}')

# Sanity check: an anomalously large component may indicate a placeholder key merging distinct patients.
LIMIAR_SUSPEITO = 10
suspeitos = adm_por_pac[adm_por_pac > LIMIAR_SUSPEITO]
if len(suspeitos):
    print(f'\n⚠️  {len(suspeitos)} patient_id(s) with more than {LIMIAR_SUSPEITO} admissions:')
    print(adm[adm['patient_id'].isin(suspeitos.index)][['patient_id'] + MERGE_KEYS].drop_duplicates().to_string(index=False))
else:
    print(f'Sanity check: no patient_id with more than {LIMIAR_SUSPEITO} admissions.')
for col in MERGE_KEYS:
    vazios = adm[col].isna().sum() + adm[col].astype(str).str.strip().isin(['','0']).sum()
    print(f'{col}: {vazios} valores nulos/placeholder')


In [ ]:
# par_unico and is_indice for ALL admissions (not only the index admission).
# is_indice = True for each patient_id's earliest admission (the index admission),
# ordered by adm_datetime — exactly the Phase 1 criterion.
pair_counts = adm.groupby(MERGE_KEYS).size().rename('n_par')
adm = adm.merge(pair_counts, on=MERGE_KEYS, how='left')
adm['par_unico'] = adm['n_par'].eq(1)
adm['ambiguo']   = adm['n_par'] > 1
adm = adm.sort_values(['patient_id','adm_datetime'], kind='stable')
adm['is_indice'] = ~adm.duplicated('patient_id', keep='first')

n_total   = len(adm)
n_indice  = int(adm['is_indice'].sum())
print(f'Total admissions (after exact duplicates): {n_total}')
print(f'index admissions:        {n_indice}'
      '(expected: 5,703 — before sheet integration and the outcome filter)')
print(f'  non-index admissions:    {n_total - n_indice}')
print(f'  unique pair:             {int(adm["par_unico"].sum())}  ({100*adm["par_unico"].mean():.1f}%)')
print(f'  ambiguous (repeated pair): {int(adm["ambiguo"].sum())}  ({100*adm["ambiguo"].mean():.1f}%)')


### Payment source (`OrigemPagamento`) — identical to Phase 1

Same derivation as Phase 1, applied here to `adm` (all admissions) instead of `index_cohort`.
Required before the merge: without it, `OrigemPagamento` does not exist in `adm` and
`preparar_boosting` fails in Phase 5 (`KeyError: ['OrigemPagamento'] not in index`).

In [ ]:
PUBLICO    = ['SESA - State Department of Health']
PARTICULAR = ['Particular']

def categorizar_convenio(valor):
    if pd.isna(valor): return 'Not reported'
    v = str(valor).strip()
    if v in PUBLICO:    return 'Public/SUS'
    if v in PARTICULAR: return 'Particular'
    return 'Private insurance'

adm['OrigemPagamento'] = adm['HealthInsuranceName'].apply(categorizar_convenio)
print('Distribution of OrigemPagamento (all admissions):')
print(adm['OrigemPagamento'].value_counts())


## 3 · Integration of the four predictor sheets — identical to Phase 1, generalized to all admissions

Same two-track logic already validated: unique pairs integrate by key; ambiguous pairs attempt recovery
by positional alignment; whatever still fails follows `AMBIGUOUS_POLICY`.

**Difference from Phase 1:** here the same key (`EpimedCode`, `MedicalRecord`) may appear multiple times
in `clean` — when a patient has more than one admission with a unique pair (the same pair legitimately
identifies both admissions). The merge therefore uses `validate='many_to_one'` instead of
`'one_to_one'`. `validate='many_to_one'` still guarantees that each row of `clean` finds at most one
matching row in the predictor sheet — protecting against silent duplication.

In [ ]:
FOLHA2_KEEP = [
    'EpimedCode','MedicalRecord','patient_id','_linha_bruta',
    'Age','Gender','Weight','Height','BMI',
    'AdmissionSourceName','AdmissionTypeName','AdmissionReasonName',
    'AdmissionMainDiagnosisName','Month Year','OrigemPagamento',
    'Charlson Comorbidity Index','MFI points','MFI Score','Frail Patient MFI',
    'Saps3Points','Saps3Death Probability Standard Equation','Sofa Score',
    'Status COVID 19',
    'mortalidade_hospitalar','mortalidade_uti','adm_datetime','par_unico','is_indice',
]
FOLHA2_KEEP = [c for c in FOLHA2_KEEP if c in adm.columns]
base_completa = adm[FOLHA2_KEEP].copy()

PRED_SHEETS = ['comorbidades','saps3_razoes','complic_dia1','fisiologia6h']

# ---- 1) Unique pair: merge by key ----
# validate='many_to_one': the same key may appear multiple times in clean
# (multiple unique-pair admissions of the same patient), but each finds
# exactly one row in the predictor sheet — guarantees no silent duplication.
clean = base_completa[base_completa['par_unico'].fillna(False)].copy()
for nome in PRED_SHEETS:
    df = raw[nome].drop_duplicates(subset=MERGE_KEYS, keep='first')
    novas = MERGE_KEYS + [c for c in df.columns if c not in clean.columns]
    clean = clean.merge(df[novas], on=MERGE_KEYS, how='left', validate='many_to_one')
clean['integravel']             = True
clean['recuperado_por_posicao'] = False

# ---- 2) Ambiguous pair: recovery by positional alignment ----
ambiguas = base_completa[~base_completa['par_unico'].fillna(False)].copy()

recuperados_linhas, falhas_linhas = [], []
for _, adm_row in ambiguas.iterrows():
    linha_alvo = adm_row['_linha_bruta']
    ec, mr     = adm_row['EpimedCode'], adm_row['MedicalRecord']
    novos_dados, ok = {}, True
    for nome in PRED_SHEETS:
        df = raw[nome]
        candidatos = df[
            (df['EpimedCode']    == ec) &
            (df['MedicalRecord'] == mr) &
            (df['_linha_bruta']  == linha_alvo)
        ]
        if len(candidatos) != 1:
            ok = False
            break
        linha = candidatos.iloc[0]
        for c in df.columns:
            if c not in FOLHA2_KEEP and c != '_linha_bruta':
                novos_dados[c] = linha[c]
    (recuperados_linhas if ok else falhas_linhas).append(
        {**adm_row.to_dict(), **novos_dados} if ok else adm_row.to_dict()
    )

recuperados    = pd.DataFrame(recuperados_linhas)
ainda_ambiguos = pd.DataFrame(falhas_linhas)
if len(recuperados):
    recuperados['integravel']             = True
    recuperados['recuperado_por_posicao'] = True
if len(ainda_ambiguos):
    ainda_ambiguos['integravel']             = False
    ainda_ambiguos['recuperado_por_posicao'] = False

print(f'Total ambiguous: {len(ambiguas)}  |  '
      f'recovered by position: {len(recuperados)}  |  '
      f'still ambiguous: {len(ainda_ambiguos)}')


### 3.1 · Safety check — swap risk between neighbors sharing the same key

Positional recovery requires that key **and** position coincide simultaneously. There is a single
scenario where this double guarantee is not enough: two admissions of the **same patient**, with the
**same full key**, sitting in **adjacent** positions in the raw spreadsheet. In that case, a neighbor
swap between Folha2 and a predictor sheet would go undetected, because the key would match either way.
This cell searches for that scenario explicitly across all ambiguous admissions.

In [ ]:
pares_risco = []
for pid, grupo in ambiguas.groupby('patient_id'):
    linhas_pid = sorted(grupo['_linha_bruta'].tolist())
    for i in range(len(linhas_pid) - 1):
        if linhas_pid[i + 1] - linhas_pid[i] == 1:
            pares_risco.append({
                'patient_id':    pid,
                'linha_bruta_a': linhas_pid[i],
                'linha_bruta_b': linhas_pid[i + 1],
            })

tab_risco = pd.DataFrame(pares_risco)

if len(tab_risco) == 0:
    print('OK — No pair of same-patient, same-key admissions in adjacent positions.')
    print('Position-recovered cases do not carry this specific cross-swap risk.')
else:
    print(f'⚠️  {len(tab_risco)} pair(s) of same-patient, same-key admissions '
          'in adjacent positions.')
    print('Positional recovery cannot, by construction, distinguish them safely.')
    print('Review manually before trusting the recovered data of these patients:')
    print(tab_risco.to_string(index=False))
    risco_path = f'{OUTDIR}/fase1b_pares_risco_troca_vizinhos.csv'
    tab_risco.to_csv(risco_path, index=False, encoding='utf-8-sig')
    print(f'List saved for review: {risco_path}')


In [ ]:
# ---- 3) Assemble the final base according to AMBIGUOUS_POLICY ----
if AMBIGUOUS_POLICY == 'exclude':
    partes = [clean]
    if len(recuperados):    partes.append(recuperados)
    todas_internacoes = pd.concat(partes, ignore_index=True)
elif AMBIGUOUS_POLICY == 'keep_folha2_only':
    partes = [clean]
    if len(recuperados):    partes.append(recuperados)
    if len(ainda_ambiguos): partes.append(ainda_ambiguos)
    todas_internacoes = pd.concat(partes, ignore_index=True)
else:
    raise ValueError(f'AMBIGUOUS_POLICY desconhecida: {AMBIGUOUS_POLICY}')

todas_internacoes = (todas_internacoes
    .sort_values(['patient_id','adm_datetime'], kind='stable')
    .reset_index(drop=True))

print(f'Combined base (policy={AMBIGUOUS_POLICY!r}): {todas_internacoes.shape}')
print(f'Pacientes distintos:        {todas_internacoes["patient_id"].nunique()}')
print(f'Index admissions in this base: {int(todas_internacoes["is_indice"].sum())}  '
      '(reference: 5,699 before the outcome filter)')


## 3.2 · Sheet-integration audit

Adapts the Phase 1 audit (§6.1) to the multiple-admissions context:
for each key present in `todas_internacoes` (unique pair), it checks whether the number of occurrences
of that key **in the predictor sheets** matches the number of occurrences in Folha2 — a necessary
condition for the key-based `merge` to be bijective and not create extra rows. Any divergence
is reported nominally for review.

In [ ]:
print('=' * 70)
print('SHEET-INTEGRATION AUDIT — PHASE 1b'.center(70))
print('=' * 70)

# Count of occurrences of each key in clean (unique pair, all admissions)
contagem_folha2 = (todas_internacoes[todas_internacoes['par_unico']]
                   .groupby(MERGE_KEYS).size())

auditoria_ok = True
for nome in PRED_SHEETS:
    contagem_aba = raw[nome].groupby(MERGE_KEYS).size()
    divergentes  = [
        k for k in contagem_folha2.index
        if contagem_aba.get(k, 0) != contagem_folha2[k]
    ]
    ok  = len(divergentes) == 0
    auditoria_ok = auditoria_ok and ok
    status = 'OK' if ok else f'⚠️  {len(divergentes)} chave(s) com contagem divergente'
    print(f'{nome:14s}: {status}')
    if not ok:
        for k in divergentes[:10]:   # mostrar no máximo 10
            print(f'    chave={k}  Folha2={contagem_folha2[k]}  {nome}={contagem_aba.get(k,0)}')
        if len(divergentes) > 10:
            print(f'    ... and {len(divergentes)-10} more key(s) — see the audit CSV.')
        audit_path = f'{OUTDIR}/fase1b_auditoria_divergencias_{nome}.csv'
        pd.DataFrame(
            [(k, int(contagem_folha2[k]), int(contagem_aba.get(k,0))) for k in divergentes],
            columns=['chave','n_folha2','n_'+nome]
        ).to_csv(audit_path, index=False, encoding='utf-8-sig')
        print(f'    Lista salva: {audit_path}')

print('-' * 70)
n_pu   = int(todas_internacoes['par_unico'].sum())
n_real = len(todas_internacoes[todas_internacoes['par_unico']])
print(f'Unique-pair admissions:                  {n_pu}')
print(f'Rows after merge (unique pair):          {n_real}')
print(f'  -> no duplication or row loss:         {n_pu == n_real}')
print('=' * 70)
if auditoria_ok and n_pu == n_real:
    print('RESULT: INTEGRATION SOUND.')
    print('Each unique-pair admission is associated with its own row')
    print('in each predictor sheet, with no duplication or loss.')
else:
    print('⚠️  RESULTADO: REVISAR — ver itens marcados acima.')
print('=' * 70)


## 4 · Restrict to known hospital outcome — same criterion as Phase 1 (Section 10)

The `mortalidade_hospitalar.notna()` filter comes **after** sheet integration and **before** feature
engineering — same order as Phase 1. This preserves traceability: cases without an outcome went through
integration, and the reason for exclusion is the unknown outcome, not an integration failure.

In [ ]:
antes = len(todas_internacoes)
n_indice_antes    = int(todas_internacoes['is_indice'].sum())
n_nao_indice_antes = antes - n_indice_antes

todas_internacoes = todas_internacoes[todas_internacoes['mortalidade_hospitalar'].notna()].copy()

n_total_depois      = len(todas_internacoes)
n_indice_depois     = int(todas_internacoes['is_indice'].sum())
n_nao_indice_depois = n_total_depois - n_indice_depois

print('Known hospital-outcome filter:')
print(f'{"":30s} {"before":>8}  {"after":>8}  {"excluded":>10}')
print('-'*60)
print(f'{"Total admissions":30s} {antes:>8}  {n_total_depois:>8}  {antes-n_total_depois:>10}')
print(f'{"Index admissions":30s} {n_indice_antes:>8}  {n_indice_depois:>8}  {n_indice_antes-n_indice_depois:>10}')
print(f'{"Non-index admissions":30s} {n_nao_indice_antes:>8}  {n_nao_indice_depois:>8}  {n_nao_indice_antes-n_nao_indice_depois:>10}')
print(f'\nHospital mortality (after filter): {todas_internacoes["mortalidade_hospitalar"].mean()*100:.1f}%')

## 5 · Feature engineering — reapplied without logic changes

The same four steps as Phase 2 (comorbidities, physiological ratios, secondary diagnoses via Folha8,
missingness indicators), now applied to `todas_internacoes` instead of the base restricted to the
index admission.

In [ ]:
COMORB_GRUPOS = {
    'comorb_cardiovascular': ['IsChfNyhaClass23','IsChfNyhaClass4','IsArterialHypertension',
        'IsAngina','IsPreviousAMI','IsCardiacArrhythmia','IsChronicAtrialFibrilation',
        'IsDeepVenousThrombosis','IsPeripheralArteryDisease'],
    'comorb_renal': ['IsCrfNoDialysis','IsCrfDialysis'],
    'comorb_hepatico': ['IsCirrhosisChildAB','IsCirrhosisChildC','IsHepaticFailure'],
    'comorb_oncologico': ['IsSolidTumorLocoregional','IsSolidTumorMetastatic',
        'IsHematologicalMalignancy','Is Chemotherapy','Is Radiation Therapy'],
    'comorb_respiratorio_cronico': ['IsSevereCopd','IsAsthma','Is History Of Pneumonia'],
    'comorb_metabolico_endocrino': ['IsDiabetesUncomplicated','IsDiabetesComplicated',
        'IsMorbidObesity','Is Hypothyroidism','Is Hyperthyroidism','Is Dyslipidemias'],
    'comorb_imunossupressao': ['IsImmunossupression','IsSteroidsUse','IsAids'],
    'comorb_transplante': ['Is Solid Organ Transplant','Is Autologous BMT','Is Allogeneic BMT',
        'Is Other Solid Organ Transplant','Is Cardiac Transplant','Is Combined Liverkidney Transplant',
        'Is Combined Pancreaskidney Transplant','Is Liver Transplant','Is Intestinal Transplant',
        'Is Pancreas Transplant','Is Lung Transplant','Is Kidney Transplant'],
    'comorb_neurologico': ['IsStrokeSequelae','IsStrokeNoSequelae','IsDementia'],
    'comorb_psiquiatrico_substancias': ['IsTobaccoConsumption','IsPsychoativeSubstanceDependence',
        'IsAlcoholism','IsPsychiatricDisease'],
    'comorb_outras': ['IsRheumaticDisease','Is Peptic Disease','IsMalnourishment'],
}
TODOS_FLAGS_COMORB = [c for grupo in COMORB_GRUPOS.values() for c in grupo]
faltando = [c for c in TODOS_FLAGS_COMORB if c not in todas_internacoes.columns]
assert not faltando, f'Comorbidity columns not found: {faltando}'

todas_internacoes[TODOS_FLAGS_COMORB] = todas_internacoes[TODOS_FLAGS_COMORB].fillna(0)
todas_internacoes['n_comorbidades']    = todas_internacoes[TODOS_FLAGS_COMORB].sum(axis=1)
for grupo, flags in COMORB_GRUPOS.items():
    todas_internacoes[grupo] = (todas_internacoes[flags].sum(axis=1) > 0).astype(int)

print('Comorbidades agrupadas.')


In [ ]:
todas_internacoes['shock_index'] = (
    todas_internacoes['Highest Heart Rate1h'] /
    todas_internacoes['Lowest Systolic Blood Pressure1h'])
todas_internacoes.loc[
    todas_internacoes['Lowest Systolic Blood Pressure1h'] <= 0, 'shock_index'] = np.nan

todas_internacoes['razao_ureia_creatinina'] = (
    todas_internacoes['Urea'] / todas_internacoes['Highest Creatinine1h'])
todas_internacoes.loc[
    todas_internacoes['Highest Creatinine1h'] <= 0, 'razao_ureia_creatinina'] = np.nan

print('Physiological ratios derived.')


In [ ]:
N_POSICOES = 12
GRANDE_AREA_MAP = {
    'Cardiovascular':'cardiovascular', 'Cardiorespiratory arrest':'cardiovascular',
    'Choque (exceto sepse)':'cardiovascular',
    'Cardiac surgeries (except congenital heart disease)':'cardiovascular',
    'Combined cardiac surgeries':'cardiovascular',
    'Cardiac surgeries for congenital heart disease':'cardiovascular',
    'Invasive cardiac and endovascular procedures':'cardiovascular',
    'Cirurgias vasculares':'cardiovascular',
    'Procedimentos / cirurgias endovasculares':'cardiovascular',
    'Congenital heart disease':'cardiovascular',
    'Respiratory (except sepsis / infection)':'respiratorio',
    'Lung, tracheal, and bronchial surgeries':'respiratorio',
    'Other thoracic surgeries':'respiratorio',
    'Infection / sepsis':'infeccioso',
    'Neurological':'neurologico', 'Neurocirurgias':'neurologico', 'Brain death':'neurologico',
    'Renal':'renal_geniturinario',
    'Male genital organ diseases':'renal_geniturinario',
    'Female genital organ diseases':'renal_geniturinario',
    'Urinary tract surgeries':'renal_geniturinario',
    'Prostate surgeries':'renal_geniturinario',
    'Gynecological and breast surgeries':'renal_geniturinario',
    'Cirurgias da mama':'renal_geniturinario', 'Breast diseases':'renal_geniturinario',
    'Obstetrics':'renal_geniturinario',
    'Obstetric surgeries and procedures':'renal_geniturinario',
    'Gastrointestinal':'gastro_hepatobiliar',
    'Liver, Biliary Tract, Spleen, and Pancreas':'gastro_hepatobiliar',
    'Abdominal / retroperitoneal surgeries':'gastro_hepatobiliar',
    'Colon, sigmoid, rectal, and anal canal surgeries':'gastro_hepatobiliar',
    'Abdominal wall hernias and defects':'gastro_hepatobiliar',
    'Gastric surgeries':'gastro_hepatobiliar',
    'Cirurgias do intestino delgado':'gastro_hepatobiliar',
    'Liver, biliary tract, and pancreas surgeries':'gastro_hepatobiliar',
    'Repair of abdominal wall hernias and defects':'gastro_hepatobiliar',
    'Bariatric surgeries':'gastro_hepatobiliar', 'Esophageal surgeries':'gastro_hepatobiliar',
    'Hematological':'onco_hematologico',
    'Tumors and oncological complications':'onco_hematologico',
    'Endocrine / metabolic':'metabolico_endocrino',
    'Endocrine gland surgeries':'metabolico_endocrino',
    'Psychiatric diseases':'psiquiatrico',
    'Causas externas':'trauma_externas', 'Non-surgical trauma':'trauma_externas',
    'Surgical polytrauma':'trauma_externas',
    'Outros / diversos':'outros',
    'Procedimentos invasivos (exceto cardiovascular)':'outros',
    'Cuidados paliativos':'outros', 'Ophthalmological':'outros', 'Outras Cirurgias':'outros',
    'Autoimmune diseases and collagenoses':'outros', 'Multiple organ dysfunction':'outros',
    'Ear, nose, and mouth diseases':'outros', 'Orthopedic surgeries':'outros',
    'Osteoarticular and soft tissue diseases':'outros',
    'Cirurgias da pele e tecidos moles':'outros', 'Head and neck surgeries':'outros',
    'Allergic and hypersensitivity reactions':'outros',
    'Skin diseases (except allergic and hypersensitivity reactions)':'outros',
    'Monitoring':'outros', 'Cirurgias de coluna':'outros',
    'Otorhinolaryngological surgeries':'outros',
}

def is_true(s):
    return s.fillna(0).astype(float).eq(1)

folha8 = raw['diag_secund']
registros = []
for i in range(1, N_POSICOES + 1):
    rcol = f'Admission Reason Name{i}'
    pcol = f'IsSecondaryDiagAtICU|Admission{i}'
    if rcol not in folha8.columns or pcol not in folha8.columns:
        continue
    presente = is_true(folha8[pcol])
    grupo_i  = folha8[rcol].map(GRANDE_AREA_MAP)
    registros.append(pd.DataFrame({
        '_linha_bruta': folha8['_linha_bruta'],
        'presente': presente,
        'grupo':    grupo_i,
    }))
long_df = pd.concat(registros, ignore_index=True)
long_df = long_df[long_df['presente']]

contagem    = long_df.groupby('_linha_bruta').size().rename('n_diag_secund_admissao')
grupos_wide = (long_df.dropna(subset=['grupo'])
               .assign(valor=1)
               .pivot_table(index='_linha_bruta', columns='grupo', values='valor',
                            aggfunc='max', fill_value=0))
grupos_wide.columns = [f'diag2_{g}' for g in grupos_wide.columns]
diag_secund_por_linha = contagem.to_frame().join(grupos_wide, how='outer').reset_index()
diag_cols = [c for c in diag_secund_por_linha.columns if c != '_linha_bruta']

# Positional verification: the Folha8 row with _linha_bruta = x must carry the same
# key as the todas_internacoes row with _linha_bruta = x.
folha8_chave_por_linha = folha8.set_index('_linha_bruta')[MERGE_KEYS]
alvo = folha8_chave_por_linha.reindex(todas_internacoes['_linha_bruta']).reset_index(drop=True)
bate = (
    (alvo['EpimedCode'].values    == todas_internacoes['EpimedCode'].values) &
    (alvo['MedicalRecord'].values == todas_internacoes['MedicalRecord'].values)
)
todas_internacoes['folha8_integravel'] = bate

n_ok    = int(bate.sum())
n_falha = int((~bate).sum())
print(f'Folha8 correctly aligned by position: {n_ok} of {len(todas_internacoes)}')
if n_falha > 0:
    print(f'⚠️  {n_falha} admission(s) misaligned in Folha8:')
    print(todas_internacoes.loc[~bate, MERGE_KEYS + ['_linha_bruta']].to_string(index=False))

todas_internacoes = todas_internacoes.merge(
    diag_secund_por_linha, on='_linha_bruta', how='left', validate='one_to_one')
todas_internacoes.loc[~todas_internacoes['folha8_integravel'], diag_cols] = 0
todas_internacoes[diag_cols] = todas_internacoes[diag_cols].fillna(0)

print(f'\nWith >=1 secondary diagnosis present: '
      f'{(todas_internacoes["n_diag_secund_admissao"]>0).sum()} '
      f'({(todas_internacoes["n_diag_secund_admissao"]>0).mean()*100:.1f}%)')


In [ ]:
VARS_INDICADOR = [
    'BMI','Height','Weight',
    'Lowest Systolic Blood Pressure1h','Lowest Diastolic Blood Pressure1h',
    'Lowest Mean Arterial Pressure1h','Highest Heart Rate1h','Highest Respiratory Rate1h',
    'Highest Temperature1h','Lowest Glasgow Coma Scale1h','Highest Leukocyte Count1h',
    'Lowest Platelets Count1h','Highest Creatinine1h','Highest Bilirubin1h',
    'Highest PH1h','Highest Pa O21h','Highest Pa CO21h','Highest Fi O21h',
    'Highest Pa O2Fi O21h','Highest Arterial Lactate1h','Urea','BUN',
    'Charlson Comorbidity Index','Sofa Score',
    'shock_index','razao_ureia_creatinina',
]
VARS_INDICADOR = [c for c in VARS_INDICADOR if c in todas_internacoes.columns]

indicadores_criados = []
for col in VARS_INDICADOR:
    if todas_internacoes[col].isna().any():
        novo = f'{col}_ausente'
        todas_internacoes[novo] = todas_internacoes[col].isna().astype(int)
        indicadores_criados.append(novo)

print(f'Missingness indicators created: {len(indicadores_criados)}')

if 'Chronic Health Status Name' in todas_internacoes.columns:
    todas_internacoes['Chronic Health Status Name'] = (
        todas_internacoes['Chronic Health Status Name']
        .astype('string').fillna('Not reported'))
    print("'Chronic Health Status Name': blanks labeled 'Not reported'.")


## 6 · Consistency check against the known cohort

Verifies that the index admissions in this base reproduce exactly the 5,676
of the Phase 1 analytical cohort. The number of non-index admissions is reported
as reference for the Phase 5 sensitivity analysis.

In [ ]:
indice_nesta_base = todas_internacoes[todas_internacoes['is_indice']]
n_indice_final    = len(indice_nesta_base)
n_nao_indice      = int((~todas_internacoes['is_indice']).sum())
referencia_ok     = (n_indice_final == 5676)

print(f'Index admissions, with known outcome:      {n_indice_final}  '
      f'(reference 5,676: {"✓ OK" if referencia_ok else "⚠️  DIVERGES"})')
print(f'Non-index admissions, with known outcome:  {n_nao_indice}')
print(f'Total: {len(todas_internacoes)}')

if not referencia_ok:
    print('\n⚠️  The index-admission count diverges from the Phase 1 reference.')
    print('Check whether the is_indice filter is being applied correctly.')

## 7 · Save the all-admissions base

In [ ]:
fase_sens_path = f'{OUTDIR}/modelo_A_todas_internacoes.csv'
todas_internacoes.to_csv(fase_sens_path, index=False, encoding='utf-8-sig')
print(f'Base salva: {fase_sens_path}')
print(f'  {todas_internacoes.shape[0]} linhas x {todas_internacoes.shape[1]} colunas')
print('\nThis file is read by Phase 5, Section 6 (sensitivity analysis) via FASE_SENS_PATH.')


---
### What this notebook delivered

- **`modelo_A_todas_internacoes.csv`** — all-admissions base (index + non-index) with known hospital
  outcome, with identity resolution, ambiguous-pair recovery, and feature engineering identical to
  Phases 1 and 2.
- **Additional columns relative to the main cohort:**
  - `is_indice` — flags each patient's index admission (True/False).
  - `folha8_integravel` — indicates whether positional alignment with Folha8 succeeded.
  - `recuperado_por_posicao` — flags ambiguous pairs recovered by positional alignment.
- **Automatic warnings** are emitted if any of the following occur:
  - Divergence in dedup propagation across sheets.
  - Anomalously large connected components (possible placeholder key).
  - Keys with divergent counts between Folha2 and predictor sheets (audit §3.2).
  - Neighbor pairs at risk of cross-swap (check §3.1).
  - Positional misalignment in Folha8.
  - Index-admission count diverging from the 5,676 reference.